In [ ]:
import pandas as pd
import os
import seaborn as sns
import re
import json
import numpy as np

In [ ]:


FILE_PATH = "../data/Scenario_6" 
TIME_WINDOW = '50ms' 
OUTPUT_FILENAME = f'../data/scenario_6_marina_features_{TIME_WINDOW}.csv'

def parse_statistics(file_path):
    """Parses the Phone_Statistics file."""
    stats_records = []
    try:
        with open(file_path, 'r', errors='ignore') as f:
            for line in f:
                try:
                    parts = line.split('::', 1)
                    if len(parts) < 2: continue
                    timestamp = float(parts[0])
                    data_prefix = 'data="'
                    json_start_index = parts[1].find(data_prefix)
                    if json_start_index == -1: continue
                    json_string = parts[1][json_start_index + len(data_prefix):-2]
                    json_data = json.loads(json_string)
                    record = {
                        'timestamp': timestamp,
                        'bwe': int(json_data.get('bwe', np.nan)),
                        'buffer_level_ms': int(json_data.get('bh', np.nan)),
                        'video_format_itag': int(json_data.get('fmt', np.nan))
                    }
                    stats_records.append(record)
                except (json.JSONDecodeError, IndexError, ValueError, KeyError):
                    continue
        if not stats_records: return None
        df = pd.DataFrame(stats_records)
        df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s')
        return df.set_index('timestamp')
    except FileNotFoundError:
        return None

def parse_tcpdump(file_path):
    """Parses a Phone_TCPdump file."""
    tcp_pattern = re.compile(
        r'^(?P<timestamp>\d+\.\d+).* length (?P<length>\d+)', re.MULTILINE
    )
    packet_records = []
    try:
        with open(file_path, 'r', errors='ignore') as f:
            # ... (rest of function is unchanged)
            # It correctly returns a DataFrame with datetime objects
            content = f.read()
            for match in tcp_pattern.finditer(content):
                packet_records.append({
                    'timestamp': float(match.group('timestamp')),
                    'length': int(match.group('length'))
                })
        if not packet_records: return None
        df = pd.DataFrame(packet_records)
        df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s')
        return df
    except FileNotFoundError:
        return None

all_processed_dfs = []
# print(f"Starting processing in: {os.path.abspath(FILE_PATH)}")
# print(f"Using time window: {TIME_WINDOW}")

for video_id in os.listdir(FILE_PATH):
    video_path = os.path.join(FILE_PATH, video_id)
    if not os.path.isdir(video_path): continue

    for iteration in os.listdir(video_path):
        iteration_path = os.path.join(video_path, iteration)
        if not os.path.isdir(iteration_path): continue
        
        
        stats_file = next((os.path.join(iteration_path, f) for f in os.listdir(iteration_path) if "Phone_Statistics" in f), None)
        tcpdump_file = next((os.path.join(iteration_path, f) for f in os.listdir(iteration_path) if "Phone_TCPdump" in f), None)

        if not stats_file or not tcpdump_file:
            print(f"    Warning: Missing files. Skipping.")
            continue
            
        stats_df = parse_statistics(stats_file)
        packets_df = parse_tcpdump(tcpdump_file)

        if stats_df is None or packets_df is None or packets_df.empty:
            print(f"    Warning: Failed to parse data. Skipping.")
            continue
        
        packets_df = packets_df.sort_values('timestamp').reset_index(drop=True)
        packets_df['ps'] = packets_df['length']
        packets_df['iat_ms'] = packets_df['timestamp'].diff().dt.total_seconds() * 1000
        packets_df['iat_ms'].fillna(0, inplace=True)
        
        packets_df['ps2'] = packets_df['ps'] ** 2
        packets_df['ps3'] = packets_df['ps'] ** 3
        packets_df['iat2'] = packets_df['iat_ms'] ** 2
        packets_df['iat3'] = packets_df['iat_ms'] ** 3
        
        packets_df.set_index('timestamp', inplace=True)
        
        agg_rules = {
            'ps': 'sum', 'ps2': 'sum', 'ps3': 'sum',
            'iat_ms': 'sum', 'iat2': 'sum', 'iat3': 'sum'
        }
        
        features_df = packets_df.resample(TIME_WINDOW).agg(agg_rules)
        features_df['packet_count'] = packets_df['ps'].resample(TIME_WINDOW).count()
        
        features_df.rename(columns={
            'ps': 'ps_sum', 'ps2': 'ps2_sum', 'ps3': 'ps3_sum',
            'iat_ms': 'iat_sum', 'iat2': 'iat2_sum', 'iat3': 'iat3_sum'
        }, inplace=True)

        merged_df = pd.merge_asof(
            left=features_df.sort_index(),
            right=stats_df.sort_index(),
            left_index=True,
            right_index=True,
            direction='backward'
        )
        
        merged_df['video_id'] = video_id
        merged_df['iteration'] = iteration
        
        all_processed_dfs.append(merged_df)


In [ ]:

if not all_processed_dfs:
    print("\nNo data was processed.")
else:
    final_df = pd.concat(all_processed_dfs)
    final_df.reset_index(inplace=True)
    
    final_df['timestamp'] = (final_df['timestamp'].astype(np.int64) // 10**6)
    
    final_df.dropna(inplace=True)
    
    feature_cols = ['packet_count', 'ps_sum', 'ps2_sum', 'ps3_sum', 'iat_sum', 'iat2_sum', 'iat3_sum']
    label_cols = ['buffer_level_ms', 'bwe', 'video_format_itag']
    id_cols = ['timestamp', 'video_id', 'iteration']
    
    for col in label_cols:
        final_df[col] = final_df[col].astype(int)
        
    final_df = final_df[id_cols + feature_cols + label_cols]

    final_df.to_csv(OUTPUT_FILENAME, index=False)

    print(f"\nProcessing complete. Data saved to '{OUTPUT_FILENAME}'")
    print(f"Total rows created: {len(final_df)}")
    print("\nFinal DataFrame Info:")
    final_df.info()
    print("\nFinal DataFrame Head:")
    print(final_df.head())

In [ ]:
#EDA
df = pd.read_csv('../data/scenario_6_marina_features_50ms.csv')

print(df.describe())

print(df.info())

sample_video_id = df['video_id'].iloc[0]
sample_iteration = df['iteration'].iloc[0]

sample_df = df[(df['video_id'] == sample_video_id) & (df['iteration'] == sample_iteration)].copy()

print(f"Visualizing a sample experiment: {sample_video_id} / {sample_iteration}")


In [ ]:
import matplotlib.pyplot as plt

df.hist(bins=50, figsize=(20, 15))
plt.tight_layout()
plt.show()

In [ ]:
# Create a figure and a set of subplots
fig, ax1 = plt.subplots(figsize=(15, 7))

# Plot Packet Size Sum (ps_sum) on the primary y-axis (ax1)
color = 'tab:blue'
ax1.set_xlabel('Timestamp (ms)')
ax1.set_ylabel('Packet Size Sum (Bytes per 50ms)', color=color)
ax1.plot(sample_df['timestamp'], sample_df['ps_sum'], color=color, label='Traffic Bursts (ps_sum)')
ax1.tick_params(axis='y', labelcolor=color)

# Create a second y-axis that shares the same x-axis
ax2 = ax1.twinx()  
color = 'tab:red'
ax2.set_ylabel('Buffer Level (ms)', color=color)  
ax2.plot(sample_df['timestamp'], sample_df['buffer_level_ms'], color=color, alpha=0.7, label='Buffer Level')
ax2.tick_params(axis='y', labelcolor=color)

# Final plot adjustments
plt.title(f'Traffic Bursts vs. Buffer Level Over Time for a Single Session')
fig.tight_layout()  # Adjust plot to prevent labels from overlapping
plt.show()

In [ ]:
# Select only the Marina feature columns for the plot
feature_cols = ['packet_count', 'ps_sum', 'ps2_sum', 'ps3_sum', 'iat_sum', 'iat2_sum', 'iat3_sum']

plt.figure(figsize=(15, 8))
sns.boxplot(data=df[feature_cols])
plt.title('Distribution of Marina Features')
plt.ylabel('Value')
plt.xlabel('Feature')

plt.xticks(rotation=45)

plt.yscale('log')
plt.show()

In [ ]:

numerical_cols = [
    'packet_count', 'ps_sum', 'ps2_sum', 'ps3_sum', 
    'iat_sum', 'iat2_sum', 'iat3_sum', 'buffer_level_ms', 'bwe'
]

# Create the heatmap
plt.figure(figsize=(12, 10))
sns.heatmap(df[numerical_cols].corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Matrix for Marina-Style Features')
plt.tight_layout()
plt.show()